In [1]:
pip install pytesseract pillow pymupdf paddleocr

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ------------- -------------------------- 6.8/19.8 MB 35.6 MB/s eta 0:00:01
   ------------------------------ --------- 14.9/19.8 MB 36.9 MB/s eta 0:00:01
   ---------------------------------------- 19.8/19.8 MB 32.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------------------------- -- 2.1/2.2 MB 32.7 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ------------------------ --------------- 7.9/12.8 MB 39.8 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 33.8 MB/s  0:00:00
   ---------------------------------------- 0.0/45.5 MB ? eta -:--:--
   ------- -------------------------------- 8.1/45.5 MB 40.2 MB/s eta 0:00:01
   -------------- ----------

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\jshin\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\modelscope\\msdatasets\\dataset_cls\\custom_datasets\\image_quality_assessment_degradation\\image_quality_assessment_degradation_dataset.py'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\jshin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print(pytesseract.get_tesseract_version())

5.5.3.20260724


In [6]:
pip install pytesseract pillow pymupdf paddleocr easyocr

Defaulting to user installation because normal site-packages is not writeable
  Using cached paddleocr-3.7.0-py3-none-any.whl.metadata (28 kB)
  Using cached paddlex-3.7.2-py3-none-any.whl.metadata (80 kB)
  Using cached modelscope-1.40.0-py3-none-any.whl.metadata (43 kB)
Using cached paddleocr-3.7.0-py3-none-any.whl (146 kB)
Using cached paddlex-3.7.2-py3-none-any.whl (2.2 MB)
Using cached modelscope-1.40.0-py3-none-any.whl (6.0 MB)

   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------------------------------------- 0/3 [modelscope]
   ---------

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\jshin\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\modelscope\\msdatasets\\dataset_cls\\custom_datasets\\image_quality_assessment_degradation\\image_quality_assessment_degradation_dataset.py'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\jshin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import numpy as np

In [13]:
from pathlib import Path

import numpy as np
import pymupdf
import pytesseract
import easyocr

from PIL import Image


# --------------------------------------------------
# Project folders
# --------------------------------------------------

DATASET_FOLDER = Path("New Dataset")
RESULT_FOLDER = Path("ocr_benchmark")

RESULT_FOLDER.mkdir(exist_ok=True)


# Tesseract is installed here on Windows
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)


# --------------------------------------------------
# Documents we want to compare
# --------------------------------------------------

TEST_DOCUMENTS = {
    "invoice": (
        DATASET_FOLDER
        / "Invoices"
        / "20251118_000612.jpg"
    ),

    "balance_sheet": (
        DATASET_FOLDER
        / "Balance Sheet"
        / "Consolidated Balance Sheet 2018.pdf"
    ),

    "profit_loss": (
        DATASET_FOLDER
        / "Profit & Loss"
        / "Consolidated Profit & Loss 2018.pdf"
    ),

    "cash_flow": (
        DATASET_FOLDER
        / "Cash Flows"
        / "Consolidated Cash Flow Statement 2025.pdf"
    ),

    "native_text_pdf": (
        DATASET_FOLDER
        / "Cash Flows"
        / "Consolidated Cash Flow Statement 2022.pdf"
    ),
}


# --------------------------------------------------
# Convert a document into images
# --------------------------------------------------

def get_images(file_path):
    """
    JPG and PNG files are already images.

    PDF pages are converted into images so that
    the OCR engines can read them.
    """

    if file_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:

        image = Image.open(file_path).convert("RGB")

        return [image]

    if file_path.suffix.lower() == ".pdf":

        pdf = pymupdf.open(file_path)

        images = []

        for page in pdf:

            # Increase the resolution before OCR.
            # This gives OCR more detail to work with.
            matrix = pymupdf.Matrix(2, 2)

            picture = page.get_pixmap(
                matrix=matrix
            )

            image = Image.frombytes(
                "RGB",
                [picture.width, picture.height],
                picture.samples
            )

            images.append(image)

        pdf.close()

        return images

    return []


# --------------------------------------------------
# Tesseract OCR
# --------------------------------------------------

def run_tesseract(images):

    all_text = []

    for page_number, image in enumerate(
        images,
        start=1
    ):

        print(
            f"    Tesseract - page {page_number}"
        )

        text = pytesseract.image_to_string(
            image
        )

        all_text.append(
            f"\n--- PAGE {page_number} ---\n\n"
            f"{text}"
        )

    return "\n".join(all_text)


# --------------------------------------------------
# EasyOCR
# --------------------------------------------------

print("Loading EasyOCR...")

easy_reader = easyocr.Reader(
    ["en"],
    gpu=False
)


def run_easyocr(images):

    all_text = []

    for page_number, image in enumerate(
        images,
        start=1
    ):

        print(
            f"    EasyOCR - page {page_number}"
        )

        # EasyOCR needs a NumPy array,
        # not a PIL Image.
        image_array = np.array(image)

        results = easy_reader.readtext(
            image_array,
            detail=1
        )

        page_text = []

        for result in results:

            text = result[1]
            confidence = result[2]

            page_text.append(
                f"{text} "
                f"[confidence: {confidence:.2f}]"
            )

        all_text.append(
            f"\n--- PAGE {page_number} ---\n\n"
            + "\n".join(page_text)
        )

    return "\n".join(all_text)


# --------------------------------------------------
# Save OCR output
# --------------------------------------------------

def save_result(
    document_name,
    ocr_name,
    text
):

    folder = RESULT_FOLDER / ocr_name

    folder.mkdir(
        exist_ok=True
    )

    output_file = (
        folder
        / f"{document_name}.txt"
    )

    output_file.write_text(
        text,
        encoding="utf-8"
    )

    print(
        f"    Saved: {output_file}"
    )


# --------------------------------------------------
# Check whether a PDF already contains text
# --------------------------------------------------

def check_native_pdf_text(file_path):

    if file_path.suffix.lower() != ".pdf":
        return False

    pdf = pymupdf.open(file_path)

    total_text = ""

    for page in pdf:
        total_text += page.get_text()

    pdf.close()

    return len(total_text.strip()) > 50


# --------------------------------------------------
# Run the benchmark
# --------------------------------------------------

print("\nChecking documents...\n")


for document_name, file_path in TEST_DOCUMENTS.items():

    if not file_path.exists():

        print(
            f"Missing file: {file_path}"
        )

        continue

    print(
        f"Found: {file_path}"
    )


print("\nStarting OCR benchmark...\n")


for document_name, file_path in TEST_DOCUMENTS.items():

    if not file_path.exists():

        continue

    print("\n" + "=" * 60)

    print(
        f"DOCUMENT: {document_name}"
    )

    print(
        f"FILE: {file_path}"
    )

    print("=" * 60)


    # Check whether this PDF has native text.
    if file_path.suffix.lower() == ".pdf":

        has_native_text = (
            check_native_pdf_text(
                file_path
            )
        )

        if has_native_text:

            print(
                "Native PDF text: YES"
            )

        else:

            print(
                "Native PDF text: NO"
            )


    # Convert document into images
    images = get_images(
        file_path
    )

    print(
        f"Pages: {len(images)}"
    )


    # --------------------------------------------------
    # Tesseract
    # --------------------------------------------------

    print("\nRunning Tesseract...")

    tesseract_text = run_tesseract(
        images
    )

    save_result(
        document_name,
        "tesseract",
        tesseract_text
    )


    # --------------------------------------------------
    # EasyOCR
    # --------------------------------------------------

    print("\nRunning EasyOCR...")

    easyocr_text = run_easyocr(
        images
    )

    save_result(
        document_name,
        "easyocr",
        easyocr_text
    )


print("\n" + "=" * 60)
print("OCR BENCHMARK COMPLETE")
print("=" * 60)

print(
    "\nResults saved to:"
)

print(
    RESULT_FOLDER.resolve()
)

Using CPU. Note: This module is much faster with a GPU.


Loading EasyOCR...

Checking documents...

Found: New Dataset\Invoices\20251118_000612.jpg
Found: New Dataset\Balance Sheet\Consolidated Balance Sheet 2018.pdf
Found: New Dataset\Profit & Loss\Consolidated Profit & Loss 2018.pdf
Found: New Dataset\Cash Flows\Consolidated Cash Flow Statement 2025.pdf
Found: New Dataset\Cash Flows\Consolidated Cash Flow Statement 2022.pdf

Starting OCR benchmark...


DOCUMENT: invoice
FILE: New Dataset\Invoices\20251118_000612.jpg
Pages: 1

Running Tesseract...
    Tesseract - page 1
    Saved: ocr_benchmark\tesseract\invoice.txt

Running EasyOCR...
    EasyOCR - page 1


C:\Users\jshin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


    Saved: ocr_benchmark\easyocr\invoice.txt

DOCUMENT: balance_sheet
FILE: New Dataset\Balance Sheet\Consolidated Balance Sheet 2018.pdf
Native PDF text: NO
Pages: 1

Running Tesseract...
    Tesseract - page 1
    Saved: ocr_benchmark\tesseract\balance_sheet.txt

Running EasyOCR...
    EasyOCR - page 1
    Saved: ocr_benchmark\easyocr\balance_sheet.txt

DOCUMENT: profit_loss
FILE: New Dataset\Profit & Loss\Consolidated Profit & Loss 2018.pdf
Native PDF text: NO
Pages: 1

Running Tesseract...
    Tesseract - page 1
    Saved: ocr_benchmark\tesseract\profit_loss.txt

Running EasyOCR...
    EasyOCR - page 1
    Saved: ocr_benchmark\easyocr\profit_loss.txt

DOCUMENT: cash_flow
FILE: New Dataset\Cash Flows\Consolidated Cash Flow Statement 2025.pdf
Native PDF text: NO
Pages: 2

Running Tesseract...
    Tesseract - page 1
    Tesseract - page 2
    Saved: ocr_benchmark\tesseract\cash_flow.txt

Running EasyOCR...
    EasyOCR - page 1
    EasyOCR - page 2
    Saved: ocr_benchmark\easyocr\cash